# SWR 沿传输线如何变化？假设有一个 10 MHz 的信号源，通过一根长度为 $L$ 的传输线为天线（负载为 $Z_L$）供电。根据 SWR 计的位置，读数会是多少？本笔记本的灵感来自于以下文献：Steve Stearns 在 2010 年 ARRL 太平洋天线研讨会上发表的《关于实际传输线上 SWR、反射功率和功率传输的事实》，链接为：["Facts About SWR, Reflected Power, and Power Transfer on Real Transmission Lines with Loss"](https://www.fars.k6ya.org/docs/Facts-about-SWR-and-Loss.pdf)。<img src="Impedance_matching_4.svg">

让我们使用 `scikit-rf` 来解决这个问题。但首先，进行常规的 Python 模块导入：

In [ ]:
%matplotlib inline

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

import skrf as rf
from skrf.constants import c
from skrf.tlineFunctions import voltage_current_propagation, zl_2_swr, zl_2_zin


In [ ]:
rf.stylely()

## 无损传输线我们从一个无损传输线开始，其传播常数为 $\gamma=j\beta$，特性阻抗为 $z_0=50\Omega$（实数）。

In [ ]:
freq = rf.Frequency(10, unit='MHz', npoints=1)

In [ ]:
# load and line properties
Z_L = 75  # Ohm
Z_0 = 50  # Ohm
L = 50  # m

# propagation constant
beta = freq.w/c
gamma = 1j*beta

下面，我们将计算传输线的电压驻波比（SWR），该值是关于传输线长度 $z$ 的函数，其中 $z$ 是从负载端测量的距离（负载端为 $z=0$，源端为 $z=L$）。

In [ ]:
z = np.linspace(start=L, stop=0, num=301)
SWRs = zl_2_swr(z0=Z_0, zl=zl_2_zin(Z_0, Z_L, gamma*z))

In [ ]:
fig, ax = plt.subplots()
ax.plot(z, SWRs, lw=2)
ax.set_xlabel('z [m]')
ax.set_ylabel('SWR')
ax.set_title('SWR along the (lossless) line')
ax.invert_xaxis()
ax.axvline(0, lw=8, color='k')
ax.axvline(L, lw=8, color='k')
ax.annotate('Load', xy=(0, 1.55), xytext=(10, 1.575),
            arrowprops=dict(facecolor='black', shrink=0.05),
            )
ax.annotate('Source', xy=(50, 1.55), xytext=(40, 1.575),
            arrowprops=dict(facecolor='black', shrink=0.05),
            )

正如预期的那样，在传输线的每个位置，驻波比都是相同的，因为正向和反向波的振幅在传输线上的每个位置也是相同的。

## 有损传输线让我们以之前的例子为例，但这次是在有损传输线上。该传输线由传播常数 $\gamma=\alpha + j\beta$ 定义：

In [ ]:
alpha = 0.01  # Np/m. Here a dummy value, just for the sake of the example
gamma = alpha + 1j*beta

In [ ]:
z = np.linspace(0, L, num=101)
SWRs = zl_2_swr(z0=Z_0, zl=zl_2_zin(Z_0, Z_L, gamma*z))

In [ ]:
fig, ax = plt.subplots()
ax.plot(z, SWRs, lw=2)
ax.set_xlabel('z [m]')
ax.set_ylabel('SWR')
ax.set_title('SWR along the (lossy) line')
ax.invert_xaxis()
ax.axvline(0, lw=8, color='k')
ax.axvline(L, lw=8, color='k')
ax.annotate('Load', xy=(0, 1.15), xytext=(10, 1.2),
            arrowprops=dict(facecolor='black', shrink=0.05),
            )
ax.annotate('Source', xy=(50, 1.4), xytext=(40, 1.5),
            arrowprops=dict(facecolor='black', shrink=0.05),
            )

对于有损耗的传输线，在负载端 SWR（驻波比）达到最大值，并在源端处降至最小值。让我们看看阻抗沿传输线如何变化：

In [ ]:
Zins = zl_2_zin(Z_0, Z_L, gamma*z)

fig, ax = plt.subplots()
ax.plot(z, np.abs(Zins/Z_0), lw=2, label='$Z/z_0$')
ax.plot(z, SWRs, lw=2, ls='--', label=r'$SWR$')
ax.plot(z, 1/SWRs, lw=2, ls='--', label=r'$1/SWR$')

ax.set_xlabel('z [m]')
ax.set_ylabel('Z (normalized to $z_0$)')
ax.set_title('Impedance along the line')
ax.invert_xaxis()
ax.axvline(0, lw=8, color='k')
ax.axvline(L, lw=8, color='k')
ax.annotate('Load', xy=(0, 1.2), xytext=(10, 1.275),
            arrowprops=dict(facecolor='black', shrink=0.05),
            )
ax.annotate('Source', xy=(50, 0.6), xytext=(40, 0.7),
            arrowprops=dict(facecolor='black', shrink=0.05),
            )
ax.legend()

前面的结果是由于电压和电流在传输线上沿其长度方向变化所致：

In [ ]:
V_s = 1
Z_in = zl_2_zin(Z_0, Z_L, gamma*z)
# Z_s = Z_0
V_in = V_s * Z_in/(Z_0 + Z_in)
I_in = V_in/(Z_0 + Z_in)
# note that here we are going from source to load
V, I = voltage_current_propagation(V_in, I_in, Z_0, gamma*z)

In [ ]:
fig, ax = plt.subplots(2,1,sharex=True)
ax[0].plot(z, np.abs(V), lw=2)
ax[1].plot(z, np.abs(I), lw=2, color='C1')
ax[1].set_xlabel('z [m]')
ax[0].set_ylabel('|V| (V)')
ax[1].set_ylabel('|I| (A)')
ax[0].set_title('Voltage')
ax[1].set_title('Current')
[a.axvline(0, lw=8, color='k') for a in ax]
[a.axvline(L, lw=8, color='k') for a in ax]
ax[1].annotate('Load', xy=(50, 0.0075), xytext=(40, 0.0075),
            arrowprops=dict(facecolor='black', shrink=0.05))
ax[1].annotate('Source', xy=(0, 0.001), xytext=(10, 0.001),
            arrowprops=dict(facecolor='black', shrink=0.05))